In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [2]:
DATA_PATH = "../data/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (6362620, 11)


In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [5]:
print(df["isFraud"].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [6]:
df = df.drop(
    columns=[
        "nameOrig",
        "nameDest"
    ]
)

print(df.shape)

(6362620, 9)


In [10]:
df["balanceDiffOrig"] = (
    df["oldbalanceOrg"] -
    df["newbalanceOrig"]
)

df[
    [
        "oldbalanceOrg",
        "newbalanceOrig",
        "balanceDiffOrig"
    ]
].head()

,oldbalanceOrg,newbalanceOrig,balanceDiffOrig
0,170136.0,160296.36,9839.64
1,21249.0,19384.72,1864.28
2,181.0,0.00,181.00
3,181.0,0.00,181.00
4,41554.0,29885.86,11668.14


In [11]:
df["balanceDiffDest"] = (
    df["newbalanceDest"] -
    df["oldbalanceDest"]
)

df[
    [
        "oldbalanceDest",
        "newbalanceDest",
        "balanceDiffDest"
    ]
].head()

,oldbalanceDest,newbalanceDest,balanceDiffDest
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,21182.0,0.0,-21182.0
4,0.0,0.0,0.0


In [12]:
df["amountToOrigBalance"] = (
    df["amount"] /
    (df["oldbalanceOrg"] + 1)
)

df["amountToDestBalance"] = (
    df["amount"] /
    (df["oldbalanceDest"] + 1)
)

df[
    [
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "balanceDiffOrig",
        "balanceDiffDest",
        "amountToOrigBalance",
        "amountToDestBalance"
    ]
].head()

In [13]:
df = pd.get_dummies(
    df,
    columns=["type"],
    drop_first=True,
    dtype=int
)

In [14]:
print(df.columns.tolist())

['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'balanceDiffOrig', 'balanceDiffDest', 'amountToOrigBalance', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [16]:
X = df.drop(
    columns=["isFraud"]
)

y = df["isFraud"]

In [18]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6362620, 14)
y shape: (6362620,)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 5090096
Testing samples : 1272524


In [21]:
print("Training distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64

Testing distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64


In [22]:
X_test.to_csv(
    "../data/X_test.csv",
    index=False
)

y_test.to_csv(
    "../data/y_test.csv",
    index=False
)

print("Test data saved successfully.")

Test data saved successfully.


In [24]:
train_data = X_train.copy()

train_data["isFraud"] = y_train.values

fraud_train = train_data[
    train_data["isFraud"] == 1
]

genuine_train = train_data[
    train_data["isFraud"] == 0
]

print("Fraud training transactions:", len(fraud_train))
print("Genuine training transactions:", len(genuine_train))

Fraud training transactions: 6570
Genuine training transactions: 5083526


In [25]:
genuine_sample = genuine_train.sample(
    n=min(
        len(genuine_train),
        len(fraud_train) * 10
    ),
    random_state=42
)

print(
    "Selected genuine transactions:",
    len(genuine_sample)
)

Selected genuine transactions: 65700


In [27]:
training_subset = pd.concat(
    [
        fraud_train,
        genuine_sample
    ]
)

training_subset = training_subset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(
    training_subset["isFraud"].value_counts()
)

isFraud
0    65700
1     6570
Name: count, dtype: int64


In [29]:
X_train_small = training_subset.drop(
    columns=["isFraud"]
)

y_train_small = training_subset["isFraud"]

print("X_train_small:", X_train_small.shape)
print("y_train_small:", y_train_small.shape)

X_train_small: (72270, 14)
y_train_small: (72270,)


In [30]:
smote = SMOTE(
    sampling_strategy=0.5,
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_small,
    y_train_small
)

print("Before SMOTE:")
print(y_train_small.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
isFraud
0    65700
1     6570
Name: count, dtype: int64

After SMOTE:
isFraud
0    65700
1    32850
Name: count, dtype: int64


In [31]:
X_train_smote.to_csv(
    "../data/X_train_smote.csv",
    index=False
)

y_train_smote.to_csv(
    "../data/y_train_smote.csv",
    index=False
)

print("SMOTE training data saved successfully.")

SMOTE training data saved successfully.


In [32]:
print("================================")
print("PREPROCESSING COMPLETE")
print("================================")

print(
    "Original dataset:",
    df.shape
)

print(
    "Training before SMOTE:",
    X_train_small.shape
)

print(
    "Training after SMOTE:",
    X_train_smote.shape
)

print(
    "Testing:",
    X_test.shape
)

PREPROCESSING COMPLETE
Original dataset: (6362620, 15)
Training before SMOTE: (72270, 14)
Training after SMOTE: (98550, 14)
Testing: (1272524, 14)
